# La IA aprende a ver · imágenes satelitales

**Caso del curso (economía agrícola)**

**La pregunta:** ¿se pueden medir las vías de un municipio desde el satélite? El caso busca la densidad vial de los municipios de Boyacá; en la sesión anterior surgió la idea de detectar las vías en imágenes satelitales con redes neuronales. Hoy probamos la primera pieza: **enseñarle a una red a reconocer qué hay en una imagen satelital**.

**La idea que se lleva:** con la IA se arma en minutos un clasificador de imágenes. La misma técnica sirve en otras disciplinas:
- **Física:** clasificar fuentes en mapas del cielo en rayos gamma o separar señal de ruido en un detector.
- **Biología:** reconocer células en microscopía, granos de polen o animales en fotos de cámaras trampa.
- **Ingeniería:** detectar grietas en concreto o defectos en piezas.
- **Humanidades:** ordenar documentos escaneados o reconocer letra manuscrita.

Lo que no cambia de una disciplina a otra es la pregunta de esta sesión: **¿cómo sé que el modelo vio lo que yo creo que vio?**

**Los datos (reales, no simulados):** EuroSAT RGB, 27.000 imágenes del satélite Sentinel-2 de 64×64 píxeles. Cada píxel mide 10 m, así que cada imagen cubre unos 640 × 640 m. Están etiquetadas en 10 clases de uso del suelo y se tomaron **en Europa**.
Fuente: Helber, P., Bischke, B., Dengel, A. y Borth, D. (2019). EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 12(7), 2217–2226. https://doi.org/10.1109/JSTARS.2019.2918242

### Antes de empezar: active la GPU gratis
En el menú de Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**. Con la GPU, entrenar toma uno o dos minutos; sin ella, mucho más. Si Colab dice que no hay GPU disponible, el código corre igual en CPU, pero más lento.

In [ ]:
# Cargar EuroSAT desde Hugging Face y mirar nueve ejemplos de cada clase
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

ds = load_dataset("timm/eurosat-rgb")          # partes: train (16.200), validation (5.400), test (5.400)
NOMBRES = ds["train"].features["label"].names  # nombres originales, en inglés
ES = {"AnnualCrop": "Cultivo anual", "Forest": "Bosque", "HerbaceousVegetation": "Vegetación herbácea",
      "Highway": "Carretera", "Industrial": "Industrial", "Pasture": "Pasto",
      "PermanentCrop": "Cultivo permanente", "Residential": "Residencial", "River": "Río", "SeaLake": "Lago o mar"}
CLASES_ES = [ES[n] for n in NOMBRES]
print(ds)

rng = np.random.default_rng(7)
etiquetas = np.array(ds["train"]["label"])
fig, ejes = plt.subplots(2, 5, figsize=(15, 6.8))
for k, eje in enumerate(ejes.flat):
    mosaico = np.full((196, 196, 3), 255, dtype=np.uint8)       # 3×3 imágenes con borde blanco
    for j, i in enumerate(rng.choice(np.flatnonzero(etiquetas == k), 9, replace=False)):
        f, c = divmod(j, 3)
        mosaico[f*66:f*66+64, c*66:c*66+64] = np.asarray(ds["train"][int(i)]["image"].convert("RGB"))
    eje.imshow(mosaico)
    eje.set_title(CLASES_ES[k], fontsize=13, fontweight="bold")
    eje.axis("off")
fig.suptitle("EuroSAT · nueve ejemplos por clase (Sentinel-2, 64×64 píxeles, 10 m por píxel)", fontsize=14)
plt.tight_layout()
plt.show()

## Cómo trabajar

1. Active la GPU (arriba) y corra la celda anterior: las imágenes se cargan solas y quedan en la variable `ds`.
2. Abra el panel de Gemini en Colab (botón de Gemini, arriba a la derecha o abajo de la celda) y pegue los prompts **uno por uno**, en orden.
3. **Lea el código antes de correrlo.** Si no entiende una línea, pregúntele a Gemini qué hace antes de aceptarla.

## Los tres prompts

**K2-img · Pida el clasificador**

```text
Con estas imágenes, entrena un clasificador que reconozca las 10 clases usando transferencia de aprendizaje con una red ya entrenada (por ejemplo, ResNet18), con una parte de las imágenes para entrenar y otra que el modelo nunca vea para probar. Escribe el código, córrelo en la GPU y dime la exactitud en las imágenes de prueba.
```

**K3-img · Interrogue el resultado**

```text
Antes de que yo lo crea: muéstrame la matriz de confusión y diez imágenes mal clasificadas. ¿Qué clases confunde y por qué? Si quiero medir la densidad vial de un municipio, ¿qué error me causaría esa confusión?
```

**K4-img · Escriba el alcance**

```text
Este modelo aprendió con imágenes de Europa. ¿Qué tendría que hacer para confiar en él en Boyacá? Escribe qué puedo afirmar y qué no.
```

## La pista (ábrala solo si se atasca)

<details><summary>Ver la pista</summary>

Mire la **fila Carretera** de la matriz de confusión: ¿a qué clase se van sus errores? ¿Qué tienen en común una carretera y un río vistos desde el espacio? Y piense **dónde se tomaron las imágenes**: ¿se parecen los campos, los techos y las vías de Europa a los de Boyacá? ¿Cuántos píxeles de ancho tiene una vía terciaria si cada píxel mide 10 m?

</details>

## Para su bitácora

```text
La IA dijo: __ · La trampa era: __ · La verifiqué con: __ · Puedo afirmar: __ · No puedo afirmar: __
```

## Si Gemini se atasca: la solución

El código completo, en cuatro pasos. Corre en menos de 3 minutos en una GPU T4 gratis: dos épocas de **ajuste fino** de una ResNet18 que ya aprendió a ver con ImageNet (millones de fotos comunes). Entrena con la parte `train` (16.200 imágenes) y mide la exactitud en la parte `test` (5.400 imágenes que el modelo nunca vio). Necesita haber corrido la celda de carga de arriba.

In [ ]:
# Paso 1 · Pasar las imágenes a tensores y elegir el dispositivo (GPU si hay)
import time, random
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

SEMILLA = 2026
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
disp = torch.device("cuda" if torch.cuda.is_available() else
                    "mps" if torch.backends.mps.is_available() else "cpu")
print("Dispositivo:", disp)

def a_tensores(parte):
    X = np.stack([np.asarray(im.convert("RGB")) for im in parte["image"]])   # (N, 64, 64, 3)
    y = np.array(parte["label"])
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(y)

t0 = time.time()
X_ent, y_ent = a_tensores(ds["train"])   # para entrenar
X_pru, y_pru = a_tensores(ds["test"])    # para probar: el modelo nunca las ve
print(f"Entrenamiento: {len(y_ent)} imágenes · Prueba: {len(y_pru)} imágenes · {time.time() - t0:.0f} s")

In [ ]:
# Paso 2 · Ajuste fino de una ResNet18 ya entrenada (transferencia de aprendizaje)
LADO, EPOCAS, LOTE = 128, 2, 128      # las imágenes de 64×64 se agrandan a 128×128
media = torch.tensor([0.485, 0.456, 0.406], device=disp).view(1, 3, 1, 1)   # normalización de ImageNet
desv = torch.tensor([0.229, 0.224, 0.225], device=disp).view(1, 3, 1, 1)

def preparar(xb, aumentar=False):
    xb = xb.to(disp).float().div(255)
    if aumentar:                       # volteos al azar: una imagen volteada sigue siendo de la misma clase
        if random.random() < 0.5: xb = xb.flip(3)
        if random.random() < 0.5: xb = xb.flip(2)
    xb = F.interpolate(xb, size=LADO, mode="bilinear", align_corners=False)
    return (xb - media) / desv

modelo = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
modelo.fc = nn.Linear(modelo.fc.in_features, 10)    # capa final nueva: 10 clases
modelo = modelo.to(disp)

opt = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
pasos = EPOCAS * ((len(y_ent) + LOTE - 1) // LOTE)
plan = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=pasos)
usar_amp = disp.type == "cuda"                       # media precisión solo en GPU NVIDIA
escalador = torch.amp.GradScaler("cuda", enabled=usar_amp)

t0 = time.time()
for epoca in range(EPOCAS):
    modelo.train()
    orden = torch.randperm(len(y_ent))
    for i in range(0, len(orden), LOTE):
        idx = orden[i:i + LOTE]
        xb, yb = preparar(X_ent[idx], aumentar=True), y_ent[idx].to(disp)
        with torch.autocast(device_type=disp.type, dtype=torch.float16, enabled=usar_amp):
            perdida = F.cross_entropy(modelo(xb), yb)
        opt.zero_grad(set_to_none=True)
        escalador.scale(perdida).backward()
        escalador.step(opt); escalador.update(); plan.step()
    print(f"Época {epoca + 1}/{EPOCAS} · pérdida del último lote {perdida.item():.3f} · {time.time() - t0:.0f} s")

In [ ]:
# Paso 3 · Exactitud en las imágenes de prueba y matriz de confusión
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

modelo.eval()
trozos = []
with torch.no_grad(), torch.autocast(device_type=disp.type, dtype=torch.float16, enabled=usar_amp):
    for i in range(0, len(y_pru), 256):
        trozos.append(modelo(preparar(X_pru[i:i + 256])).argmax(1).cpu())
pred, real = torch.cat(trozos).numpy(), y_pru.numpy()
print(f"Exactitud en prueba: {(pred == real).mean():.1%} ({(pred == real).sum()} de {len(real)} imágenes)")

mc = confusion_matrix(real, pred, labels=list(range(10)))
fig, eje = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(mc, display_labels=CLASES_ES).plot(ax=eje, cmap="Blues", xticks_rotation=45, colorbar=False)
eje.set_xlabel("Clase que predijo el modelo"); eje.set_ylabel("Clase real")
eje.set_title("Matriz de confusión · imágenes de prueba")
plt.tight_layout(); plt.show()

errores = [(CLASES_ES[i], CLASES_ES[j], int(mc[i, j])) for i in range(10) for j in range(10) if i != j and mc[i, j] > 0]
print(pd.DataFrame(errores, columns=["Real", "Predicha", "Imágenes"])
        .sort_values("Imágenes", ascending=False).head(8).to_string(index=False))
car, rio = NOMBRES.index("Highway"), NOMBRES.index("River")
print(f"\nCarretera → Río: {mc[car, rio]} de {mc[car].sum()} carreteras · Río → Carretera: {mc[rio, car]} de {mc[rio].sum()} ríos")

In [ ]:
# Paso 4 · Ver los errores: carreteras leídas como río y ríos leídos como carretera
grupos = [(np.flatnonzero((real == car) & (pred == rio)), "Real: Carretera\nDijo: Río"),
          (np.flatnonzero((real == rio) & (pred == car)), "Real: Río\nDijo: Carretera")]
n = max(1, min(8, max(len(g) for g, _ in grupos)))
fig, ejes = plt.subplots(2, n, figsize=(2 * n, 5), squeeze=False)
for fila, (indices, titulo) in enumerate(grupos):
    for k in range(n):
        eje = ejes[fila, k]; eje.axis("off")
        if k < len(indices):
            eje.imshow(X_pru[indices[k]].permute(1, 2, 0).numpy())
            eje.set_title(titulo, fontsize=9)
    if len(indices) == 0:
        ejes[fila, 0].text(0.5, 0.5, "(ninguna)", ha="center", va="center", fontsize=11)
fig.suptitle(f"Carretera ↔ Río · {len(grupos[0][0])} carreteras leídas como río, {len(grupos[1][0])} ríos leídos como carretera",
             fontsize=12)
plt.tight_layout(); plt.show()

**Antes de usar este modelo en Boyacá**, tres preguntas que la exactitud no contesta:
- **Dónde aprendió.** Todas las imágenes son de Europa: otros cultivos, otros techos, otras vías. La exactitud de arriba vale para Europa; en Boyacá hay que medirla de nuevo con imágenes de Boyacá etiquetadas a mano.
- **Qué ve.** Cada píxel mide 10 m: una vía terciaria de pocos metros de ancho ocupa menos de un píxel. La clase «Carretera» de EuroSAT son sobre todo vías grandes.
- **Qué mide.** El modelo le pone **una etiqueta a un cuadro de 640 × 640 m**; no cuenta kilómetros de vía. Para la densidad vial hace falta otra tarea (segmentar la vía píxel a píxel) o cruzar con un mapa de vías.